In [6]:
import requests
import pandas as pd
import pytz

def converter_data_trello(data_str):
    if not data_str or pd.isna(data_str):
        return pd.NaT
    dt_utc = pd.to_datetime(data_str, utc=True, errors='coerce')
    if pd.isna(dt_utc):
        return pd.NaT
    return dt_utc.tz_convert('America/Sao_Paulo')

NOME_QUADRO = "CTI "

AUTH = {
    "key": "94e7ea17c0af203c3719162f3a019b8b",
    "token": "ATTA64a25b844de4470670bb44db434a1cd9285d4186778dedafe131f041aa8dcd31CFB2E264"
}

def trello_get(endpoint, params=None):
    url = f"https://api.trello.com/1/{endpoint}"
    response = requests.get(url, params={**AUTH, **(params or {})})
    response.raise_for_status()
    return response.json()

print("=" * 60)
print("🚀 INICIANDO ETL DO TRELLO")
print("=" * 60)

# 1. Buscar quadro
boards = trello_get("members/me/boards")
board = next((b for b in boards if b['name'] == NOME_QUADRO), None)
if not board:
    raise Exception(f"Quadro '{NOME_QUADRO}' não encontrado")

print(f"✅ Conectado: {board['name']}")

# 2. Buscar listas
lists = trello_get(f"boards/{board['id']}/lists")
id_para_nome_lista = {lista['id']: lista['name'] for lista in lists}
print(f"✅ {len(lists)} listas encontradas")

# 3. Buscar cards
cards = trello_get(f"boards/{board['id']}/cards")
print(f"✅ {len(cards)} cards encontrados")

# 4. Buscar ações
all_actions_create = trello_get(f"boards/{board['id']}/actions", params={"filter": "createCard"})
all_actions_update = trello_get(f"boards/{board['id']}/actions", params={"filter": "updateCard"})
print(f"✅ {len(all_actions_create)} ações de criação")
print(f"✅ {len(all_actions_update)} ações de movimentação")

# Mapa de criação
card_criacao = {}
for action in all_actions_create:
    if action['type'] == 'createCard':
        card_id = action['data']['card']['id']
        card_criacao[card_id] = converter_data_trello(action['date'])

# =========================================
# PROCESSAR AÇÕES (df_acoes)
# =========================================
acoes_list = []
for action in all_actions_update:
    card_id = action['data']['card']['id']
    card = next((c for c in cards if c['id'] == card_id), {})
    
    acoes_list.append({
        'card_id': card_id,
        'card_name': card.get('name'),
        'action_date': converter_data_trello(action['date']),
        'list_before': action['data'].get('listBefore', {}).get('name'),
        'list_after': action['data'].get('listAfter', {}).get('name'),
        'action_type': action['type']
    })

# =========================================
# PROCESSAR CARDS (df_cards)
# =========================================
cards_list = []
for card in cards:
    data_criacao = card_criacao.get(card['id'])
    if pd.isna(data_criacao):
        data_criacao = converter_data_trello(card.get('dateLastActivity'))
    
    cards_list.append({
        'card_id': card['id'],
        'card_name': card['name'],
        'card_url': card.get('url'),
        'data_criacao': data_criacao,
        'data_ultima_atividade': converter_data_trello(card.get('dateLastActivity')),
        'data_inicio': converter_data_trello(card.get('start')),
        'data_vencimento': converter_data_trello(card.get('due')),
        'id_list': card.get('idList'),
        'closed': card.get('closed'),
        'labels': str(card.get('labels', [])),
        'desc': card.get('desc', '')[:100]
    })

# =========================================
# PROCESSAR LEAD TIME (df_lead)
# =========================================
lead_times_list = []
for card in cards:
    card_id = card['id']
    card_name = card['name']
    
    data_criacao = card_criacao.get(card_id)
    if pd.isna(data_criacao):
        data_criacao = converter_data_trello(card.get('dateLastActivity'))
    
    data_ultima_atividade = converter_data_trello(card.get('dateLastActivity'))
    lista_atual = id_para_nome_lista.get(card.get('idList'), '')
    
    if lista_atual == 'Concluido':
        status = 'Concluído'
        data_conclusao = data_ultima_atividade
        lead_time_dias = (data_conclusao - data_criacao).days if pd.notna(data_criacao) and pd.notna(data_conclusao) else None
    else:
        status = 'Não concluído'
        data_conclusao = pd.NaT
        lead_time_dias = None
    
    lead_times_list.append({
        'card_id': card_id,
        'card_name': card_name,
        'status': status,
        'lista_atual': lista_atual,
        'data_criacao': data_criacao,
        'data_ultima_atividade': data_ultima_atividade,
        'data_conclusao': data_conclusao,
        'lead_time_dias': lead_time_dias,
    })

# =========================================
# CRIAR DATAFRAMES
# =========================================
df_acoes = pd.DataFrame(acoes_list)
df_cards = pd.DataFrame(cards_list)
df_lead = pd.DataFrame(lead_times_list)

print("\n" + "=" * 60)
print("📊 DATAFRAMES CRIADOS")
print("=" * 60)
print(f"df_acoes: {len(df_acoes)} linhas")
print(f"df_cards: {len(df_cards)} linhas")
print(f"df_lead: {len(df_lead)} linhas")

# =========================================
# VERIFICAR CARDS NÃO CONCLUÍDOS
# =========================================
nao_concluidos = df_lead[df_lead['status'] == 'Não concluído']
print(f"\n📊 CARDS NÃO CONCLUÍDOS: {len(nao_concluidos)}")
for idx, row in nao_concluidos.head(10).iterrows():
    print(f"   ❌ {row['card_name']}")

# =========================================
# EXPORTAR CSVs
# =========================================
print("\n📁 Exportando CSVs...")
df_lead.to_csv('lead_time.csv', index=False, sep=';', date_format='%Y-%m-%d %H:%M:%S')
df_acoes.to_csv('acoes_trello.csv', index=False, sep=';', date_format='%Y-%m-%d %H:%M:%S')
df_cards.to_csv('cards_trello.csv', index=False, sep=';', date_format='%Y-%m-%d %H:%M:%S')
print("   ✅ lead_time.csv")
print("   ✅ acoes_trello.csv")
print("   ✅ cards_trello.csv")

# =========================================
# =========================================
# EXPORTAR EXCEL (SOLUÇÃO GARANTIDA)
# =========================================
print("\n📁 Exportando Excel...")

try:
    from openpyxl import Workbook
    
    # Criar cópias sem timezone
    df_lead_excel = df_lead.copy()
    df_cards_excel = df_cards.copy()
    df_acoes_excel = df_acoes.copy()
    
    # Remover timezone de todas as colunas de data
    for df in [df_lead_excel, df_cards_excel, df_acoes_excel]:
        for col in df.columns:
            if 'data' in col.lower() or 'date' in col.lower() or 'action' in col.lower():
                if pd.api.types.is_datetime64_any_dtype(df[col]):
                    try:
                        df[col] = pd.to_datetime(df[col]).dt.tz_localize(None)
                    except:
                        pass
    
    with pd.ExcelWriter('analise_trello.xlsx', engine='openpyxl') as writer:
        
        if not df_lead_excel.empty:
            df_lead_excel.to_excel(writer, sheet_name='Lead Time', index=False)
            print("   ✅ Aba 'Lead Time' criada")
        
        if not df_cards_excel.empty:
            df_cards_excel.to_excel(writer, sheet_name='Cards', index=False)
            print("   ✅ Aba 'Cards' criada")
        
        if not df_acoes_excel.empty:
            df_acoes_excel.to_excel(writer, sheet_name='Ações', index=False)
            print("   ✅ Aba 'Ações' criada")
        
        if not df_acoes.empty:
            pivot_listas = pd.crosstab(
                df_acoes['card_name'],
                df_acoes['list_after'],
                normalize=False
            )
            pivot_listas.to_excel(writer, sheet_name='Pivot_Listas')
            print("   ✅ Aba 'Pivot_Listas' criada")
    
    print("\n✅ analise_trello.xlsx gerado com sucesso!")

except Exception as e:
    print(f"   ⚠️ Erro ao gerar Excel: {e}")
    print("   Os CSVs foram gerados normalmente.")

🚀 INICIANDO ETL DO TRELLO
✅ Conectado: CTI 
✅ 7 listas encontradas
✅ 20 cards encontrados
✅ 20 ações de criação
✅ 50 ações de movimentação

📊 DATAFRAMES CRIADOS
df_acoes: 50 linhas
df_cards: 20 linhas
df_lead: 20 linhas

📊 CARDS NÃO CONCLUÍDOS: 0

📁 Exportando CSVs...
   ✅ lead_time.csv
   ✅ acoes_trello.csv
   ✅ cards_trello.csv

📁 Exportando Excel...
   ✅ Aba 'Lead Time' criada
   ✅ Aba 'Cards' criada
   ✅ Aba 'Ações' criada
   ✅ Aba 'Pivot_Listas' criada

✅ analise_trello.xlsx gerado com sucesso!
